# EP.03.02. Streaming Facturas: Procesamiento con Ficheros en Apache Spark
**Materia:** Manejo Masivo de Datos  
**Tecnología:** Apache Spark 3.5.3 · PySpark Structured Streaming (File Source)  
**Entorno:** Ubuntu Linux · Python 3.12 · JupyterLab  
### Objetivo
Procesar archivos JSON de facturas generados por distintas sucursales, descomponiendo la lista anidada de artículos (`InvoiceLineItems`) en filas individuales planas, primero en **modo Batch** y posteriormente adaptado a **Structured Streaming** con `maxFilesPerTrigger`, `cleanSource`, `checkpointLocation` y `trigger`.
## 1. Versión Inicial (Batch): Lectura, Transformación y Guardado

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Streaming Facturas")
    .master("local[2]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Como el notebook está dentro de invoices/, leemos directamente desde ./ENTRADA/
raw_df = spark.read.json("./ENTRADA/*.json")

raw_df.printSchema()
raw_df.show(5)

root
 |-- CESS: double (nullable = true)
 |-- CGST: double (nullable = true)
 |-- CashierID: string (nullable = true)
 |-- CreatedTime: long (nullable = true)
 |-- CustomerCardNo: string (nullable = true)
 |-- CustomerType: string (nullable = true)
 |-- DeliveryAddress: struct (nullable = true)
 |    |-- AddressLine: string (nullable = true)
 |    |-- City: string (nullable = true)
 |    |-- ContactNumber: string (nullable = true)
 |    |-- PinCode: string (nullable = true)
 |    |-- State: string (nullable = true)
 |-- DeliveryType: string (nullable = true)
 |-- InvoiceLineItems: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- ItemCode: string (nullable = true)
 |    |    |-- ItemDescription: string (nullable = true)
 |    |    |-- ItemPrice: double (nullable = true)
 |    |    |-- ItemQty: long (nullable = true)
 |    |    |-- TotalValue: double (nullable = true)
 |-- InvoiceNumber: string (nullable = true)
 |-- NumberOfItems: long (nullable = t

In [3]:
from pyspark.sql.functions import expr

# 1. Descomponer (explode) el array de artículos en filas individuales
explode_df = raw_df.selectExpr(
    "InvoiceNumber",
    "CreatedTime",
    "StoreID",
    "PosID",
    "CustomerType",
    "PaymentMethod",
    "DeliveryType",
    "DeliveryAddress.City",
    "DeliveryAddress.State",
    "DeliveryAddress.PinCode",
    "explode(InvoiceLineItems) as LineItem"
)

# 2. Aplanar los campos de LineItem con nombres claros
flattened_df = (
    explode_df
    .withColumn("ItemCode", expr("LineItem.ItemCode"))
    .withColumn("ItemDescription", expr("LineItem.ItemDescription"))
    .withColumn("ItemPrice", expr("LineItem.ItemPrice"))
    .withColumn("ItemQty", expr("LineItem.ItemQty"))
    .withColumn("TotalValue", expr("LineItem.TotalValue"))
    .drop("LineItem")
)

flattened_df.printSchema()
flattened_df.show(10, truncate=False)

# 3. Guardar el resultado Batch en ./SALIDA en formato JSON
(
    flattened_df.write
    .format("json")
    .mode("overwrite")
    .save("./SALIDA")
)

print("Escritura Batch completada en ./SALIDA")

root
 |-- InvoiceNumber: string (nullable = true)
 |-- CreatedTime: long (nullable = true)
 |-- StoreID: string (nullable = true)
 |-- PosID: string (nullable = true)
 |-- CustomerType: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- DeliveryType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- PinCode: string (nullable = true)
 |-- ItemCode: string (nullable = true)
 |-- ItemDescription: string (nullable = true)
 |-- ItemPrice: double (nullable = true)
 |-- ItemQty: long (nullable = true)
 |-- TotalValue: double (nullable = true)

+-------------+-------------+-------+------+------------+-------------+-------------+---------+--------------+-------+--------+-------------------+---------+-------+----------+
|InvoiceNumber|CreatedTime  |StoreID|PosID |CustomerType|PaymentMethod|DeliveryType |City     |State         |PinCode|ItemCode|ItemDescription    |ItemPrice|ItemQty|TotalValue|
+-------------+-----------

---
## 2. Versión Streaming: Procesamiento Continuo de Facturas desde Ficheros

In [4]:
# 1. Definir el esquema estático a partir de los datos ya leídos (obligatorio en file streaming)
invoice_schema = raw_df.schema

# 2. Lectura en Streaming desde ./ENTRADA/ procesando 1 archivo por micro-lote
# y archivando automáticamente los archivos ya procesados en ./ARCHIVO
stream_raw_df = (
    spark.readStream
    .format("json")
    .schema(invoice_schema)
    .option("maxFilesPerTrigger", 1)
    .option("cleanSource", "archive")
    .option("sourceArchiveDir", "./ARCHIVO")
    .load("./ENTRADA")
)

# 3. Transformación: descomponer InvoiceLineItems y aplanar columnas
stream_explode_df = stream_raw_df.selectExpr(
    "InvoiceNumber",
    "CreatedTime",
    "StoreID",
    "PosID",
    "CustomerType",
    "PaymentMethod",
    "DeliveryType",
    "DeliveryAddress.City",
    "DeliveryAddress.State",
    "DeliveryAddress.PinCode",
    "explode(InvoiceLineItems) as LineItem"
)

stream_flattened_df = (
    stream_explode_df
    .withColumn("ItemCode", expr("LineItem.ItemCode"))
    .withColumn("ItemDescription", expr("LineItem.ItemDescription"))
    .withColumn("ItemPrice", expr("LineItem.ItemPrice"))
    .withColumn("ItemQty", expr("LineItem.ItemQty"))
    .withColumn("TotalValue", expr("LineItem.TotalValue"))
    .drop("LineItem")
)

# 4. Escritura continua en ./SALIDA con checkpoint para tolerancia a fallos y trigger de 10 segundos
invoice_writer_query = (
    stream_flattened_df.writeStream
    .format("json")
    .queryName("Flattened Invoice Writer")
    .outputMode("append")
    .option("path", "./SALIDA")
    .option("checkpointLocation", "./chk-point-dir")
    .trigger(processingTime="10 seconds")
    .start()
)

print("Streaming de facturas iniciado. Esperando archivos en ./ENTRADA/ ...")
invoice_writer_query.awaitTermination()

Streaming de facturas iniciado. Esperando archivos en ./ENTRADA/ ...


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [5]:
# Verificar los datos aplanados generados por el stream en ./SALIDA
resultado_stream_df = spark.read.json("./SALIDA")
print(f"Total de artículos (filas) procesados en streaming: {resultado_stream_df.count()}")
resultado_stream_df.show(10, truncate=False)

# Detener streams activos y cerrar Spark limpiamente
for q in spark.streams.active:
    q.stop()
spark.stop()
print("Sesión de Spark cerrada correctamente.")

Total de artículos (filas) procesados en streaming: 3990
+-----------------+-------------+------------+-------------+-------------+--------+--------------------+---------+-------+-------------+-------+------+---------+-------+----------+
|City             |CreatedTime  |CustomerType|DeliveryType |InvoiceNumber|ItemCode|ItemDescription     |ItemPrice|ItemQty|PaymentMethod|PinCode|PosID |State    |StoreID|TotalValue|
+-----------------+-------------+------------+-------------+-------------+--------+--------------------+---------+-------+-------------+-------+------+---------+-------+----------+
|Bokaro Steel City|1595689270697|PRIME       |HOME-DELIVERY|94201418     |258     |Closet              |1687.0   |2      |CASH         |509723 |POS365|Jharkhand|STR7443|3374.0    |
|Bokaro Steel City|1595689270697|PRIME       |HOME-DELIVERY|94201418     |538     |Grandmother clock   |1301.0   |1      |CASH         |509723 |POS365|Jharkhand|STR7443|1301.0    |
|Bokaro Steel City|1595689270697|PRIME

---
## 3. Tolerancia a Fallos y Garantías en Structured Streaming
* **Checkpoint (`checkpointLocation`):** Spark guarda el progreso exacto en `./chk-point-dir/` mediante cuatro subdirectorios clave:
  * `offsets/`: registra qué archivos de entrada (`Invoice-set1.json`, `Invoice-set2.json`, `Invoice-set3.json`) fueron asignados a cada micro-lote (`0`, `1`, `2`).
  * `commits/`: confirma qué micro-lotes terminaron de escribirse con éxito en el destino.
  * `sources/` y `metadata/`: almacenan el identificador único del flujo y el estado de los ficheros leídos.
* **Garantía *Exactly-Once* (Exactamente una vez):** Si el proceso falla o se detiene a mitad de una ejecución, al reiniciar el *query* apuntando al mismo `checkpointLocation`, Spark consulta `offsets/` y `commits/` para retomar exactamente desde el último lote no confirmado, evitando tanto la pérdida de facturas como la duplicación de registros en `./SALIDA`.
* **Gestión de Archivos Fuente (`cleanSource="archive"`):** Una vez confirmado el procesamiento de cada archivo JSON, Spark lo traslada automáticamente a `./ARCHIVO`, manteniendo `./ENTRADA` limpia y evitando reprocesamientos innecesarios.